# D · Phase 2: can the LLM learn an examiner's habits from 3 past applications?

Tasks: Disposition, Remaining Rounds, Time to Next Action. Protocol: `D_trajectory/personalization_protocol.md`.

Setups (protocol section 2):
- LLM, no examiner block; LLM + full card, own / wrong examiner: **done 2026-11-03, results reused, not re-run**.
- LLM + full event histories of 3 past applications (no numbers calculated), own / wrong examiner: **run here**.
- No LLM (computed in Step 5): numbers only (card numbers blended with the art-unit pool, K estimated from the data);
  plain math on the same 3 histories.

**My Drive must hold** (shortcuts are fine: right-click the file in the team folder → **Organize → Add shortcut → My Drive**):
- the `D_trajectory` folder;
- `data_v1_share_v1.2.zip`;
- the team package `B_D_C_phase1_v4_reviewed.zip` (the notebook checks its sha256);
- A's `profile_corpus_v2.zip` (the notebook builds the 3-application histories from it once and checks the file's sha1).

**How to run**
1. Step 0: first time `STAGE = 'formatcheck'` (5 cases, about $0.30 for GPT).
2. Secrets (key icon, Notebook access on): `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_DEPLOYMENT` (or `OPENAI_API_KEY`); Qwen: `QWEN_BASE_URL`, `QWEN_AZURE_API_KEY`.
3. **Runtime → Run all.** If the format check passes, set `STAGE = 'full'` and **Run all** again (the 5 checked cases come from the cache).


In [8]:
# Step 0: choices
RUN_GPT = True      # GPT-5-mini, reasoning_effort low (about $5 for own + mismatched, both prompts)
RUN_QWEN = False    # Qwen3.6-27B team setting: needs the live Azure deployment (billed per hour). Delete it when done.
STAGE = 'full'   # 'formatcheck' = 5 fixed cases (protocol section 7), 'full' = all 100 test_core cases
SETTINGS = ['own', 'mismatched']
CHECK_IDS = '13060040,13484627,13627444,13577408,14031839'

In [9]:
# Step 1: find the files on Drive, unzip, build the examiner histories, check hashes, install the tools
from google.colab import drive
drive.mount('/content/drive')
import glob, hashlib, json, os, subprocess, sys
def find(name):
    return (glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True) +
            glob.glob(f'/content/drive/.shortcut-targets-by-id/**/{name}', recursive=True))
roots = glob.glob('/content/drive/MyDrive/**/D_trajectory/code/run_trajectory_llm.py', recursive=True)
zips, pkgs, corp = find('data_v1_share_v1.2.zip'), find('B_D_C_phase1_v4_reviewed.zip'), find('profile_corpus_v2.zip')
assert roots, 'Cannot find the D_trajectory folder in My Drive.'
assert zips, 'Cannot find data_v1_share_v1.2.zip in My Drive.'
assert pkgs, 'Cannot find B_D_C_phase1_v4_reviewed.zip. In Drive: right-click it in the team folder → Organize → Add shortcut → My Drive.'
ROOT = os.path.dirname(os.path.dirname(roots[0]))
need = ['code/examiner_history.py', 'code/build_examiner_histories.py', 'code/history_math.py', 'code/credibility_k.py',
        'code/examiner_prior.py', 'code/compare_history_runs.py', 'code/copy_check_cell.py',
        'code/run_trajectory_llm.py', 'code/run_rounds_next_llm.py', 'code/score_trajectory.py', 'code/score_rounds_next.py',
        'code/traj_common.py', 'prompts/trajectory_forecast_profile.txt', 'prompts/rounds_next_forecast_profile.txt',
        'inputs/state_predictions_A.jsonl', 'inputs/trajectory_inputs.jsonl', 'personalization_protocol.md',
        'outputs/D/trajectory_predictions_gpt-5-mini_state.jsonl', 'outputs/D/rounds_next_predictions_gpt-5-mini_state.jsonl',
        'outputs/D/trajectory_predictions_km.jsonl', 'outputs/D/rounds_next_predictions_km.jsonl',
        'outputs/D/trajectory_predictions_gpt-5-mini_state_profile_own.jsonl', 'outputs/D/rounds_next_predictions_gpt-5-mini_state_profile_own.jsonl',
        'outputs/D/trajectory_predictions_gpt-5-mini_state_profile_mismatched.jsonl', 'outputs/D/rounds_next_predictions_gpt-5-mini_state_profile_mismatched.jsonl']
missing = [n for n in need if not os.path.exists(os.path.join(ROOT, n))]
assert not missing, f'Missing in D_trajectory on Drive: {missing}'

# frozen hashes (protocol section 3)
PKG_SHA256 = 'd1e6e358757ba758a17373f14894449284253da1cfe4d775a0bbac6a7b207d8f'
PROMPTS = {'trajectory_forecast_profile.txt': 'd7e75c380ac8', 'rounds_next_forecast_profile.txt': '8656a89b9b39'}
HIST_SHA1 = '5e5eaaa9c839787c01895f48b0c3bb314df44fba'
got = hashlib.sha256(open(pkgs[0], 'rb').read()).hexdigest()
assert got == PKG_SHA256, f'Profile package is not the frozen v4 file (sha256 {got}).'
for f, h in PROMPTS.items():
    g = hashlib.sha1(open(os.path.join(ROOT, 'prompts', f), 'rb').read()).hexdigest()[:12]
    assert g == h, f'Prompt {f} changed ({g}, frozen {h}). Do not edit prompts after the protocol.'

DATA = '/content/data_v1_share'
if not os.path.exists(DATA + '/labels.csv'):
    subprocess.run(['unzip', '-q', '-o', zips[0], '-d', '/content/'], check=True)
PKG = '/content/B_D_C_phase1_v4_reviewed'
if not os.path.exists(PKG + '/RELEASE_MANIFEST.json'):
    subprocess.run(['unzip', '-q', '-o', pkgs[0], '-d', '/content/'], check=True)
assert os.path.exists(PKG + '/RELEASE_MANIFEST.json'), 'Package unzipped to an unexpected folder.'
STATE = os.path.join(ROOT, 'inputs', 'state_predictions_A.jsonl')
HIST = os.path.join(ROOT, 'inputs', 'examiner_histories_v1.jsonl')
if not os.path.exists(HIST):
    assert corp, 'Cannot find profile_corpus_v2.zip. In Drive: right-click it in the team folder → Organize → Add shortcut → My Drive.'
    subprocess.run([sys.executable, os.path.join(ROOT, 'code', 'build_examiner_histories.py'), '--pkg', PKG,
                    '--corpus', corp[0], '--out', HIST], check=True)
got = hashlib.sha1(open(HIST, 'rb').read()).hexdigest()
assert got == HIST_SHA1, f'examiner_histories_v1.jsonl differs from the frozen file ({got}). Send this to Claude.'
print('ROOT =', ROOT); print('DATA =', DATA); print('PKG  =', PKG)
!pip -q install "openai==2.54.0" "scikit-survival==0.28.0"
%cd {ROOT}/code
print('Step 1 OK')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT = /content/drive/MyDrive/D_trajectory
DATA = /content/data_v1_share
PKG  = /content/B_D_C_phase1_v4_reviewed
/content/drive/MyDrive/D_trajectory/code
Step 1 OK


In [10]:
# Step 2: read the secrets (never typed into the notebook) and test the connection
from google.colab import userdata
def get(k):
    try:
        return (userdata.get(k) or '').strip()
    except Exception:
        return ''
def sh(args):
    p = subprocess.Popen([sys.executable, '-u'] + args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = []
    for line in p.stdout:
        print(line, end=''); out.append(line)
    p.wait()
    return p.returncode, ''.join(out)
if RUN_GPT:
    if get('AZURE_OPENAI_ENDPOINT') and get('AZURE_OPENAI_API_KEY'):   # same route as the earlier runs
        for k in ['AZURE_OPENAI_ENDPOINT', 'AZURE_OPENAI_API_KEY', 'AZURE_OPENAI_DEPLOYMENT']:
            assert get(k), f'Secret {k} is missing'
            os.environ[k] = get(k)
        os.environ.pop('OPENAI_API_KEY', None)
        print('GPT route: Azure, deployment', os.environ['AZURE_OPENAI_DEPLOYMENT'])
    else:
        assert get('OPENAI_API_KEY'), 'Add the Azure secrets or OPENAI_API_KEY.'
        for k in ['AZURE_OPENAI_ENDPOINT', 'AZURE_OPENAI_API_KEY']:
            os.environ.pop(k, None)
        os.environ['OPENAI_API_KEY'] = get('OPENAI_API_KEY'); os.environ['AZURE_OPENAI_DEPLOYMENT'] = 'gpt-5-mini'
        print('GPT route: OpenAI API, model gpt-5-mini (the earlier runs used Azure; note this in the report)')
    rc, out = sh(['run_trajectory_llm.py', '--provider', 'azure', '--root', ROOT, '--data', DATA, '--ping'])
    assert 'connection ok' in out, 'GPT test failed. Send the message above to Claude.'
if RUN_QWEN:
    for k in ['QWEN_BASE_URL', 'QWEN_AZURE_API_KEY']:
        assert get(k), f'Secret {k} is missing'
        os.environ[k] = get(k)
    os.environ['QWEN_DEPLOYMENT'] = get('QWEN_DEPLOYMENT') or 'qwen--qwen36-27b'
    rc, out = sh(['run_trajectory_llm.py', '--provider', 'qwen', '--root', ROOT, '--data', DATA, '--ping'])
    assert 'connection ok' in out, 'Qwen test failed. Send the message above to Claude.'
print('Step 2 OK')

GPT route: Azure, deployment gpt-5-mini
connection ok: {"ok": true} | model: gpt-5-mini-2025-08-07
Step 2 OK


In [11]:
# Step 3: run the two new setups (3 full histories, own and wrong examiner). Same case record and A state (A_v1) as all runs.
def run(script, provider, setting, extra):
    args = [script, '--provider', provider, '--root', ROOT, '--data', DATA, '--split', 'core',
            '--state', STATE, '--state-version', 'A_v1', '--profile-pkg', PKG,
            '--examiner-input', 'history', '--examiner-setting', setting] + extra
    return sh(args)
jobs = [p for p, ok in (('azure', RUN_GPT), ('qwen', RUN_QWEN)) if ok]
for provider in jobs:
    for setting in SETTINGS:
        for script in ['run_trajectory_llm.py', 'run_rounds_next_llm.py']:
            if STAGE == 'formatcheck':
                tag = ('qwen' if provider == 'qwen' else 'gpt') + '_' + setting + '_' + script.split('_')[1]
                extra = ['--ids', CHECK_IDS, '--workers', '1',
                         '--dump-prompts', os.path.join(ROOT, 'outputs', 'D', 'dev', 'prompts_history_' + tag)]
            else:
                extra = ['--workers', '1']   # 1 at a time: stays under the Azure tokens-per-minute limit (429)
            rc, out = run(script, provider, setting, extra)
            assert rc in (0, None), f'{provider} {setting} {script} stopped. Send the message above to Claude.'
print('Step 3 OK')

  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done
gpt-5-mini_state_history_own [core] ok 100/100  failed 0  from cache 5  new API tokens in 4,265,565 out 87,530  est. cost of this run $1.24 (at $0.25/$2.0 per 1M)
wrote /content/drive/MyDrive/D_trajectory/outputs/D/trajectory_predictions_gpt-5-mini_state_history_own.jsonl
  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done
gpt-5-mini_state_history_own [core] ok 100/100  from cache 5  tokens in 4,271,930 out 37,517  est. $1.14
wrote /content/drive/MyDrive/D_trajectory/outputs/D/rounds_next_predictions_gpt-5-mini_state_history_own.jsonl
  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done
gpt-5-mini_state_history_mismatched [core] ok 100/100  failed 0  from cache 5  new API t

In [12]:
# Step 4 (format check only): the pass rules of protocol section 7. Accuracy is NOT looked at.
import pandas as pd, re
if STAGE == 'formatcheck':
    models = [{'azure': 'gpt-5-mini', 'qwen': 'qwen3.6-27b'}[j] for j in jobs]
    rows = []
    for f in sorted(glob.glob(os.path.join(ROOT, 'outputs', 'D', 'dev', '*_state_history_*_formatcheck.jsonl'))):
        if not any(m in os.path.basename(f) for m in models):
            continue
        for line in open(f):
            r = json.loads(line)
            rows.append(dict(file=os.path.basename(f).replace('_formatcheck.jsonl', ''), case=r['sample_id'], status=r['status'],
                             finish=','.join(r.get('finish_reasons') or []), attempts=r.get('n_attempts'),
                             past_apps=r.get('n_past_apps'), prompt_tokens=r.get('prompt_tokens'), error=str(r.get('error') or '')[:80]))
    T = pd.DataFrame(rows); display(T)
    bad = T[(T.status != 'ok') | T.finish.str.contains('length')]
    # the history block must sit before "Task", carry the case's cutoff date, and hold event lists only (no statistics)
    for d in glob.glob(os.path.join(ROOT, 'outputs', 'D', 'dev', 'prompts_history_*')):
        if not any(os.path.basename(d).startswith('prompts_history_' + {'azure': 'gpt', 'qwen': 'qwen'}[j]) for j in jobs):
            continue
        for p in glob.glob(d + '/*.txt'):
            t = open(p, encoding='utf-8').read()
            i, j = t.find('Examiner history reference available before'), t.find('\nTask')
            assert 0 < i < j, f'History block missing or misplaced in {p}'
            cut = re.search(r'after the cutoff date (\d{4}-\d{2}-\d{2})', t).group(1)
            assert 'available before ' + cut in t, f'History cutoff differs from the case cutoff in {p}'
            block = t[i:j]
            assert 'Past application' in block or 'No past applications' in block, f'No past-application history in {p}'
            assert 'median' not in block and '%' not in block, f'Statistics found in the history block of {p}'
    print('One prompt, around the history block:')
    t = open(sorted(glob.glob(os.path.join(ROOT, 'outputs', 'D', 'dev', 'prompts_history_*', '13060040.txt')))[0], encoding='utf-8').read()
    i = t.find('- Examiner history reference for this'); print(t[i:i + 1800])
    print('\nFORMAT CHECK', 'PASSED' if len(bad) == 0 and len(T) == 20 * len(jobs) else 'NOT PASSED: see the table')

In [13]:
# Step 5 (full stage only): baselines without an LLM, scores, and the fixed comparisons
if STAGE == 'full':
    K_FILE = os.path.join(ROOT, 'metrics', 'D', 'credibility_k.json')
    sh(['credibility_k.py', '--episodes', PKG + '/source_D/case_episodes.csv', '--out', K_FILE])
    got = json.load(open(K_FILE))['k_used']
    assert got == {'next_log_days': 1.8, 'allowed_24': 3.3, 'abandoned_24': 8.4, 'rounds': 4.5}, f'K differs from the frozen estimate: {got}. Send this to Claude.'
    sh(['examiner_prior.py', '--root', ROOT, '--cards', PKG + '/source_D/stats_cards.jsonl', '--k-file', K_FILE])   # numbers only
    sh(['history_math.py', '--root', ROOT])                                                                          # math on the same 3 histories
    sh(['score_trajectory.py', '--root', ROOT])
    sh(['score_rounds_next.py', '--root', ROOT])
    sh(['compare_history_runs.py', '--root', ROOT])
    print('ALL DONE. Take a screenshot of this page and send it to Claude. Delete the Qwen deployment now if you used it.')

next_log_days  EPV 0.3726  VHM 0.2102  K 1.8  95% [1.4, 2.3]  (VHM<=0 in 0% of draws)  episodes 6006 | vs overall mean: K 1.7
allowed_24     EPV 0.1196  VHM 0.0368  K 3.3  95% [2.4, 4.5]  (VHM<=0 in 0% of draws)  episodes 4150 | vs overall mean: K 3.6
abandoned_24   EPV 0.0943  VHM 0.0112  K 8.4  95% [6.0, 13.2]  (VHM<=0 in 0% of draws)  episodes 4150 | vs overall mean: K 7.9
rounds         EPV 0.7880  VHM 0.1766  K 4.5  95% [3.3, 6.5]  (VHM<=0 in 0% of draws)  episodes 4023 | vs overall mean: K 3.5
K used: {'next_log_days': 1.8, 'allowed_24': 3.3, 'abandoned_24': 8.4, 'rounds': 4.5}
           method  disp_c_index  disp_ibs  rounds_mae  rounds_acc  next_c_index  next_mae_days  next_median_pred
               km         0.500     0.188        0.99        0.37         0.500          36.26              62.0
        prior_own         0.692     0.156        0.94        0.44         0.721          27.21              63.0
 prior_mismatched         0.564     0.184        1.15        0.32     

In [14]:
# Step 6 (full stage only): copy check, no API. Does the LLM just repeat a number it can read off its examiner block?
if STAGE == 'full':
    exec(open(os.path.join(ROOT, 'code', 'copy_check_cell.py')).read())

,method,compared_with,n,days_exactly_equal,days_within_3,days_rank_corr_with_ref,days_rank_corr_with_truth,allow24_rank_corr_with_card
0,gpt-5-mini + own card,card median days,100,0.860,0.910,0.983,0.601,0.971
1,gpt-5-mini + mismatched card,card median days,100,0.880,0.910,0.989,-0.040,0.921
2,gpt-5-mini + own histories,math median of the 3 histories,100,0.000,0.090,0.275,0.219,NaN
3,gpt-5-mini + mismatched histories,math median of the 3 histories,97,0.021,0.093,0.110,0.196,NaN
4,prior_own (no LLM),NaN,100,NaN,NaN,NaN,0.592,NaN
5,history_math_own (no LLM),NaN,100,NaN,NaN,NaN,0.565,NaN


GPT + own histories: share of rationales that mention the past applications: 1.0
14588659    Applicant filed an amended response contesting...
13862126    Applicant filed an amendment/response on the c...
14130025    Applicant filed an amendment and arguments add...
